# Structured Output - Type-Safe Agent Responses

## Overview
Learn how to get structured, type-safe responses from agents using Pydantic models instead of plain text.

## What You'll Learn
- Defining output schemas with Pydantic
- Using `output_type` parameter
- Automatic validation and type checking
- When to use structured vs. free-form output

## Key Concepts
- **Pydantic Models**: Define the structure and types of expected output
- **output_type**: Parameter that enforces a specific response format
- **Validation**: Automatic checking that output matches the schema
- **Type Safety**: Get predictable data structures instead of text parsing

## Step 1: Install Dependencies

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Step 2: Configure Authentication

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Step 3: Import Required Classes

Import `BaseModel` from Pydantic to define output schemas.

In [ ]:
import asyncio
from agents import Agent, Runner
from pydantic import BaseModel

## Step 4: Define Output Schema

Create a Pydantic model defining the exact structure you want.

**Benefits:**
- Type checking (str, int, list, etc.)
- Required vs optional fields
- Nested structures
- Automatic validation

💡 **Tip**: Use descriptive field names - the agent reads them!

In [ ]:
class CalendarEvent(BaseModel):
    name: str                 # Event/person name
    date: str                 # Event date
    participants: list[str]   # List of participants

## Step 5: Create Agent with output_type

Set `output_type` to your Pydantic model.

**What happens:**
- Agent is forced to return data matching the schema
- Output is automatically validated
- `result.final_output` is a Pydantic model instance, not a string
- You can access fields directly: `result.final_output.name`

In [ ]:
agent = Agent(
    name="Calendar extractor",
    instructions="Extract main person of the event, event date and participants from the text",
    model="openai.gpt-5.5",
    output_type=CalendarEvent  # Enforce structured output
)

## Step 6: Run and Get Structured Output

The agent will extract information and return a validated `CalendarEvent` object.

🎯 **Result**: Instead of parsing text, you get a typed object!

In [ ]:
result = await Runner.run(
    agent,
    "John's birthday is on 10 April 1975. All his four family members joined the party."
)

print(result.final_output)
print(f"\nEvent: {result.final_output.name}")
print(f"Date: {result.final_output.date}")
print(f"Participants: {result.final_output.participants}")

### Key Takeaways
- Structured output eliminates text parsing
- Pydantic provides automatic validation
- Use when you need predictable data formats
- Perfect for data extraction, form filling, API responses